# Balancing chemical equations (null space)

*chemistry · beginner* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/chemistry/equation_balancing)

Balancing an equation means finding coefficients $\nu_s$ that conserve every element and the charge. Written as a matrix, that is $M\nu = 0$: the coefficients are the null space of the element–species matrix. One batched SVD solves seven reactions at once, from methane combustion to permanganate in acid, and spots the one whose balance is not unique.

**Vector form**

$$M\,\boldsymbol\nu = \mathbf 0,\qquad M = C\,\mathrm{diag}(\boldsymbol\sigma),\qquad M = U\,\Sigma\,V^{\mathsf T}\;\Rightarrow\;\boldsymbol\nu \propto \mathbf v_{\min}$$

**Index (tensor) form**

$$M_{res} = C_{res}\,\sigma_{rs} \qquad M_{res}\,\nu_{rs} = 0 \qquad M_{res} = U_{rek}\,\Sigma_{rk}\,V_{rsk} \qquad \nu_{rs} = \frac{k_r}{\min_t |V_{rtS}|}\,V_{rsS}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $C_{res}$ | `composition(reactants, products)` | (E, S) | atoms of element $e$ in species $s$ of reaction $r$; row q is the charge (from the tiny formula parser) |
| $\sigma_{rs}$ | `sigma` | (S,) | $+1$ for a reactant, $-1$ for a product |
| $M_{res} = C_{res}\sigma_{rs}$ | `einsum('es,s->es', C, sigma)` | (E, S) | no index summed: each column is signed |
| $M_{res}$ | `batch(M_list)` | (R, E_max+S_max, S_max) | all reactions padded into one tensor; unused columns pinned to $\nu_s = 0$ |
| $U_{rek}\Sigma_{rk}V_{rsk}$ | `torch.linalg.svd(M)` | (R, S), (R, S, S) | batched SVD; singular values in descending order |
| $\Sigma_{rS} \approx 0$ | `null_dimension(sv)` | (R,) | how many singular values vanish: 1 → unique balance, 2 → two independent reactions |
| $\nu_{rs}$ | `to_integers(V[..., -1])` | (R, S) | last singular vector, scaled so the smallest entry is 1, then by the first $k_r$ that gives whole numbers |
| $M_{res}\nu_{rs}$ | `einsum('res,rs->re', M, nu)` | (R, E) | repeated $s$ ⇒ summed: atoms of $e$ created; zero for every element and the charge |

Each element gives one linear equation in the unknown coefficients, so $S$ species and $E$ elements give an $E\times S$ system $M\nu = 0$. Usually $M$ has rank $S-1$ and the solutions form a line: the balanced equation is unique up to a common factor. The SVD finds that line as the right singular vector with singular value 0 (the last one); dividing by its smallest entry and trying multipliers $k = 1, 2, \dots$ gives the smallest whole numbers, e.g. $\mathrm{CH_4} + 2\,\mathrm{O_2} \rightarrow \mathrm{CO_2} + 2\,\mathrm{H_2O}$. For ions, charge is just one more row: without it, $\mathrm{MnO_4^-} + \mathrm{Fe^{2+}} + \mathrm{H^+}$ would have a 2D null space, and with it the familiar $1{:}5{:}8 \rightarrow 1{:}5{:}4$ redox equation drops out. The peroxide example $\mathrm{H_2} + \mathrm{O_2} \rightarrow \mathrm{H_2O} + \mathrm{H_2O_2}$ has two zero singular values: it hides two independent reactions ($2\mathrm{H_2} + \mathrm{O_2} \rightarrow 2\mathrm{H_2O}$ and $\mathrm{H_2} + \mathrm{O_2} \rightarrow \mathrm{H_2O_2}$), and any positive mix of them balances too. Algebra alone cannot pick one; the real proportions are set by the chemistry (kinetics, conditions). The code shows the basis reactions found by setting one coefficient to zero in the 2D null space. Try a reaction of your own by adding it to `REACTIONS` in tensors.py; the parser reads brackets and charges such as 'Ca3(PO4)2' or 'Fe^3+'.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/chemistry/equation_balancing/main.py
"""Main loop: formulas → composition matrices → SVD null space → whole-number coefficients → plot."""

from pathlib import Path

import torch

from problems._common.runtime import cli, device

from . import equation, plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    names = list(tensors.REACTIONS)
    parts = [tensors.composition(*tensors.REACTIONS[n], dev) for n in names]  # (C, σ, elements, species) each
    M_list = [equation.signed_composition(C, sigma) for C, sigma, _, _ in parts]
    M, n_species = tensors.batch(M_list)  # (R, E_max + S_max, S_max)
    print(f"{len(names)} reactions in one batch: M has shape {tuple(M.shape)}")

    sv, V = solve.null_vectors(M)  # (R, S), (R, S, S)
    nu, ok = solve.to_integers(V[..., -1], params["max_scale"])  # smallest singular vector of each reaction
    dims = equation.null_dimension(sv)  # (R,)
    residual = equation.imbalance(M, nu)  # (R, E_max + S_max): must be all zero
    sv_real = [torch.linalg.svdvals(M[r, :, :s]) for r, s in enumerate(n_species)]  # without the padding columns

    balanced, plane = [], []
    for r, name in enumerate(names):
        species = parts[r][3]
        if dims[r] == 1 and ok[r]:
            text = _equation_text(nu[r, : n_species[r]], species, len(tensors.REACTIONS[name][0]))
            balanced.append((name, nu[r, : n_species[r]]))
            print(f"{name:<15} {text}   (max |Δ| = {residual[r].abs().max().item():.0f})")
        else:
            print(f"{name:<15} null space of dimension {dims[r].item()}: no unique balance")
            N = V[r, : n_species[r], -dims[r] :]  # (S, 2) basis of the null space
            cands = solve.reactions_in_plane(N)
            n_left = len(tensors.REACTIONS[name][0])
            for c in cands.unique(dim=0).flip(0):
                if solve.physical(c) and (c != 0).sum() > 1:
                    plane.append(c)
                    print(f"{'':<15}   basis reaction: {_equation_text(c, species, n_left)}")
            if len(plane) == 2:
                mix = plane[0] + plane[1]
                print(f"{'':<15}   any positive mix also balances, e.g. {_equation_text(mix, species, n_left)}")
    print("element and charge conservation M ν = 0 holds for every reaction:", bool((residual[dims == 1] == 0).all()))

    show = params["show"]
    r = names.index(show)
    C, sigma, elements, species = parts[r]
    nu_show = nu[r, : n_species[r]] if dims[r] == 1 else plane[0]
    return plot.figure(M_list[r], nu_show, elements, species, show, names, sv_real, balanced, plane, out_dir)


def _equation_text(nu, species: list[str], n_left: int) -> str:
    def side(idx):
        terms = [(f"{int(nu[i])} " if nu[i] != 1 else "") + tensors.pretty(species[i]) for i in idx if nu[i] != 0]
        return " + ".join(terms)

    return f"{side(range(n_left))} → {side(range(n_left, len(species)))}"


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/chemistry/equation_balancing/equation.py
r"""Chemistry → tensor translation.

A reaction  ν₁ A₁ + ν₂ A₂ + …  →  … + ν_S A_S  conserves every element e (and the charge):

    atoms of e on the left  =  atoms of e on the right      ⇔      M_{es} ν_s = 0   (summed over s)

with the signed composition matrix M_{es} = C_{es} σ_s, where C_{es} counts atoms of element e in species s and
σ_s = +1 for reactants, -1 for products. So the coefficients ν form the null space of M: a vector that M sends
to zero. Repeated indices are summed: s in the conservation law. A batch of reactions r just adds an index.
"""

import torch


def signed_composition(C: torch.Tensor, sigma: torch.Tensor) -> torch.Tensor:
    """M_{es} = C_{es} σ_s: products count negative.   C: (E, S), σ: (S,) → (E, S).  No index summed."""
    return torch.einsum("es,s->es", C, sigma)


def imbalance(M: torch.Tensor, nu: torch.Tensor) -> torch.Tensor:
    """Δ_{re} = M_{res} ν_{rs}: atoms (or charge) of e created by reaction r.   Repeated s ⇒ summed.

    M: (R, E, S), ν: (R, S) → (R, E). A balanced equation has Δ = 0 for every element.
    """
    return torch.einsum("res,rs->re", M, nu)


def null_dimension(sv: torch.Tensor, tol: float = 1e-9) -> torch.Tensor:
    """Number of (numerically) zero singular values = number of independent reactions.   sv: (R, S) → (R,).

    1 → the balanced equation is unique up to scale;  2 → two reactions are mixed, the balance is not unique.
    """
    return (sv < tol * sv[..., :1]).sum(-1)


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/chemistry/equation_balancing/tensors.py
"""Build the concrete tensors: parse formulas, list the reactions and stack their composition matrices.

A formula is written the usual way, with an optional charge after a caret: 'H2O', 'Ca(OH)2', 'MnO4^-', 'Fe^3+'.
The charge is stored as one more "element" called 'q', so the same matrix product checks atoms AND charge.
"""

import re

import torch

ATOM = re.compile(r"([A-Z][a-z]?)(\d*)")  # an element symbol and an optional count: 'C', 'H12', 'Ca'
GROUP_END = re.compile(r"\)(\d*)")  # a closing bracket and its multiplier: ')2'
CHARGE = re.compile(r"(\d*)([+-])")  # '3+', '-', '2-'

# name: (reactants, products)
REACTIONS = {
    "methane": (["CH4", "O2"], ["CO2", "H2O"]),
    "propane": (["C3H8", "O2"], ["CO2", "H2O"]),
    "glucose": (["C6H12O6", "O2"], ["CO2", "H2O"]),
    "photosynthesis": (["CO2", "H2O"], ["C6H12O6", "O2"]),
    "permanganate": (["MnO4^-", "Fe^2+", "H^+"], ["Mn^2+", "Fe^3+", "H2O"]),
    "phosphate": (["Ca(OH)2", "H3PO4"], ["Ca3(PO4)2", "H2O"]),
    "peroxide": (["H2", "O2"], ["H2O", "H2O2"]),  # two independent reactions hide in here
}


def parse_formula(formula: str) -> dict[str, int]:
    """Count atoms: 'Ca(OH)2' → {'Ca': 1, 'O': 2, 'H': 2};  'MnO4^-' → {'Mn': 1, 'O': 4, 'q': -1}.

    Brackets are handled with a stack: '(' opens a new tally, ')n' multiplies it by n and adds it to the outer one.
    """
    body, _, charge = formula.partition("^")
    stack = [{}]  # one tally per open bracket; stack[0] is the whole formula
    i = 0
    while i < len(body):
        if body[i] == "(":
            stack.append({})
            i += 1
        elif body[i] == ")":
            m = GROUP_END.match(body, i)
            group, times = stack.pop(), int(m.group(1) or 1)
            for element, n in group.items():
                stack[-1][element] = stack[-1].get(element, 0) + n * times
            i = m.end()
        else:
            m = ATOM.match(body, i)
            if m is None:
                raise ValueError(f"cannot read {formula!r} at position {i}")
            stack[-1][m.group(1)] = stack[-1].get(m.group(1), 0) + int(m.group(2) or 1)
            i = m.end()
    counts = stack[0]
    if charge:
        m = CHARGE.fullmatch(charge)
        counts["q"] = int(m.group(1) or 1) * (1 if m.group(2) == "+" else -1)
    return counts


def composition(reactants: list[str], products: list[str], device: torch.device):
    """Composition matrix C_es (E, S), side signs σ_s (S,), element names and species names.

    C_es = number of atoms of element e in species s (the charge row 'q' holds the charge of s);
    σ_s = +1 for a reactant and -1 for a product.
    """
    species = reactants + products
    counts = [parse_formula(f) for f in species]
    elements = sorted({e for c in counts for e in c if e != "q"})
    if any("q" in c for c in counts):
        elements.append("q")  # charge balance as one more row
    C = torch.tensor([[c.get(e, 0) for c in counts] for e in elements], dtype=torch.float64, device=device)
    sigma = torch.tensor([1.0] * len(reactants) + [-1.0] * len(products), dtype=torch.float64, device=device)
    return C, sigma, elements, species


def batch(matrices: list[torch.Tensor]) -> tuple[torch.Tensor, list[int]]:
    """Stack signed matrices M_es of different sizes into one (R, E_max + S_max, S_max) tensor.

    Missing element rows are zero (a zero row changes nothing). A missing species column would add a fake
    solution ν_s = anything, so each unused column s gets a 1 in an extra row of its own: that row says ν_s = 0.
    The extra rows also make the matrix tall, so the SVD returns one singular value per species.
    """
    E = max(M.shape[0] for M in matrices)
    S = max(M.shape[1] for M in matrices)
    out = torch.zeros(len(matrices), E + S, S, dtype=matrices[0].dtype, device=matrices[0].device)
    for r, M in enumerate(matrices):
        e, s = M.shape
        out[r, :e, :s] = M
        for j in range(s, S):
            out[r, E + j, j] = 1.0  # pin the padding coefficient to zero
    return out, [M.shape[1] for M in matrices]


def pretty(formula: str) -> str:
    """'C6H12O6' → 'C₆H₁₂O₆', 'Fe^2+' → 'Fe²⁺' for the terminal."""
    body, _, charge = formula.partition("^")
    sub = str.maketrans("0123456789", "₀₁₂₃₄₅₆₇₈₉")
    sup = str.maketrans("0123456789+-", "⁰¹²³⁴⁵⁶⁷⁸⁹⁺⁻")
    return body.translate(sub) + charge.translate(sup)


def latex(formula: str) -> str:
    """'Ca3(PO4)2' → 'Ca_{3}(PO_{4})_{2}', 'MnO4^-' → 'MnO_{4}^{-}' for matplotlib mathtext."""
    body, _, charge = formula.partition("^")
    body = re.sub(r"(\d+)", r"_{\1}", body)
    return rf"\mathrm{{{body}}}" + (f"^{{{charge}}}" if charge else "")


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/chemistry/equation_balancing/solve.py
"""Find the coefficients: SVD for the null space, then scale the vector to the smallest whole numbers."""

import torch


def null_vectors(M: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """Batched SVD M = U diag(sv) Vᵀ.   M: (R, E, S) with E ≥ S → sv (R, S) descending, V (R, S, S).

    The columns of V belong to the singular values; the last ones, with sv ≈ 0, span the null space.
    """
    _, sv, Vh = torch.linalg.svd(M, full_matrices=False)
    return sv, Vh.transpose(-1, -2)


def to_integers(v: torch.Tensor, max_scale: int = 60, tol: float = 1e-6) -> tuple[torch.Tensor, torch.Tensor]:
    """Scale null vectors to the smallest whole numbers.   v: (..., S) → (ν (..., S) integers, ok (...,) bool).

    1) flip the sign so the largest entry is positive; 2) divide by the smallest non-zero |entry| so it becomes 1;
    3) try multipliers k = 1, 2, … and keep the first one that turns every entry into a whole number.
    The smallest entry is then k and the result has no common factor left.
    """
    big = v.gather(-1, v.abs().argmax(-1, keepdim=True))  # (..., 1) the largest entry, with its sign
    v = v / big
    smallest = torch.where(v.abs() > 1e-9, v.abs(), torch.inf).amin(-1, keepdim=True)
    v = v / smallest  # smallest non-zero entry → ±1
    k = torch.arange(1, max_scale + 1, dtype=v.dtype, device=v.device)  # (K,)
    scaled = v[..., None, :] * k[:, None]  # (..., K, S)
    error = (scaled - scaled.round()).abs().amax(-1)  # (..., K) distance from whole numbers
    good = error < tol
    first = good.int().argmax(-1)  # index of the first multiplier that works (0 if none does)
    nu = scaled.gather(-2, first[..., None, None].expand(*first.shape, 1, v.shape[-1])).squeeze(-2).round()
    return nu, good.any(-1)


def reactions_in_plane(N: torch.Tensor) -> torch.Tensor:
    """For a 2D null space with basis N (S, 2): the S special combinations with one coefficient set to zero.

    W_{ts} = N_{t0} N_{s1} - N_{t1} N_{s0}  → column s is a null vector whose entry t = s vanishes.
    Returns (S, S): row s is that vector (whole numbers, sign fixed so the largest entry is positive).
    """
    W = torch.einsum("t,s->ts", N[:, 0], N[:, 1]) - torch.einsum("t,s->ts", N[:, 1], N[:, 0])
    nu, _ = to_integers(W.T)
    return nu


def physical(nu: torch.Tensor) -> torch.Tensor:
    """A reaction must have every coefficient ≥ 0 (reactants left, products right).   (..., S) → (...,) bool."""
    return (nu >= 0).all(-1)


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/chemistry/equation_balancing/plot.py
"""Static figure: the signed composition matrix of one reaction, the singular values of all, and the balanced list."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.colors import BoundaryNorm, ListedColormap

from problems._common.plotting import save_png

from . import tensors

FLOOR = 1e-17  # exact zeros drawn here on the log axis


def figure(
    M: torch.Tensor,
    nu: torch.Tensor,
    elements: list[str],
    species: list[str],
    show: str,
    names: list[str],
    sv: list[torch.Tensor],
    balanced: list,
    plane: list,
    out_dir: Path,
) -> Path:
    fig = plt.figure(figsize=(11, 6.6))
    grid = fig.add_gridspec(2, 2, width_ratios=[1.05, 1], height_ratios=[1.15, 1])
    ax_m = fig.add_subplot(grid[0, 0])
    ax_s = fig.add_subplot(grid[0, 1])
    ax_t = fig.add_subplot(grid[1, :])

    # --- (a) signed composition matrix with the coefficients underneath ---
    m = M.cpu().numpy() + 0.0  # + 0.0 turns -0 into 0
    lim = int(np.abs(m).max())
    levels = np.arange(-lim - 0.5, lim + 1.5)
    cmap = ListedColormap(plt.get_cmap("RdBu_r")(np.linspace(0.1, 0.9, len(levels) - 1)))
    ax_m.imshow(m, cmap=cmap, norm=BoundaryNorm(levels, cmap.N), aspect="auto")
    for (e, s), value in np.ndenumerate(m):
        ax_m.text(s, e, f"{value:.0f}", ha="center", va="center", fontsize=10, color="#111827")
    ax_m.set_xticks(range(len(species)), [f"${tensors.latex(f)}$\nν = {int(n)}" for f, n in zip(species, nu)])
    ax_m.set_yticks(range(len(elements)), ["charge" if e == "q" else e for e in elements])
    ax_m.grid(False)
    ax_m.set_title(f"{show}: $M_{{es}} = C_{{es}}\\,\\sigma_s$ (products < 0),  $M\\nu = 0$", fontsize=10)

    # --- (b) singular values: a zero one ⇔ a null vector ⇔ a balanced reaction ---
    for r, s in enumerate(sv):
        s = s.cpu().numpy()
        zero = s < 1e-9 * s[0]
        ax_s.plot([r] * (~zero).sum(), s[~zero], "o", color="#2563eb", ms=5)
        dx = 0.15 * (np.arange(zero.sum()) - (zero.sum() - 1) / 2)  # side by side when there are several
        ax_s.plot(r + dx, np.maximum(s[zero], FLOOR), "o", color="#dc2626", ms=7)
        if zero.sum() > 1:
            ax_s.text(r, FLOOR * 1e2, f"{zero.sum()} zeros", ha="center", fontsize=8, color="#dc2626")
    ax_s.set_yscale("log")
    ax_s.set_ylim(FLOOR / 10, 100)
    ax_s.set_xticks(range(len(names)), names, rotation=30, ha="right", fontsize=8)
    ax_s.set_ylabel("singular values of $M$")
    ax_s.set_title("red: (numerically) zero → null space", fontsize=10)

    # --- (c) the balanced equations ---
    ax_t.axis("off")
    rows = [(name, _latex_equation(nu_r, name)) for name, nu_r in balanced]
    rows += [("peroxide (basis)", _latex_equation(c, "peroxide")) for c in plane]
    for i, (name, eq) in enumerate(rows):
        y = 1 - (i + 0.5) / len(rows)
        ax_t.text(0.02, y, name, fontsize=10, color="#6b7280", va="center", transform=ax_t.transAxes)
        ax_t.text(0.24, y, f"${eq}$", fontsize=12, va="center", transform=ax_t.transAxes)
    ax_t.set_title("whole-number coefficients from the smallest singular vector", fontsize=10)

    fig.tight_layout()
    return save_png(fig, out_dir / "equation_balancing.png")


def _latex_equation(nu: torch.Tensor, name: str) -> str:
    reactants, products = tensors.REACTIONS[name]
    species = reactants + products

    def side(idx):
        terms = [(f"{int(nu[i])}\\," if nu[i] != 1 else "") + tensors.latex(species[i]) for i in idx if nu[i] != 0]
        return " + ".join(terms)

    return f"{side(range(len(reactants)))} \\rightarrow {side(range(len(reactants), len(species)))}"


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'show': 'permanganate',
    'max_scale': 60
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.chemistry.equation_balancing")]:
    del sys.modules[name]
main = importlib.import_module("problems.chemistry.equation_balancing.main")
problem_dir = Path("problems/chemistry/equation_balancing")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
